# 🌐 Módulo 12: Servidor HTTP Concurrente en C
## Notebook de Conocimiento Guiado

**Objetivo:** Entender cómo funciona un servidor HTTP/1.1 con thread pool en C,
desde los sockets POSIX hasta la sincronización con mutex y variables de condición.

El código completo está en `codigo/c/http_server.c`.

**Lore:** Eres el Ingeniero de Infraestructura Galáctica. Cada conexión TCP que llega
es una nave solicitando datos críticos. Tu thread pool la despacha sin demora.

| Sección | Concepto |
|---------|---------|
| 📚 Teoría | TCP sockets, HTTP/1.1, thread pool |
| 🔨 Demo | Parsear HTTP/1.1 en Python (mimics C) |
| 🔨 Demo | Thread pool con queue.Queue |
| ✏️ Ejercicio 1 | Parsear cabeceras completas |
| ✏️ Ejercicio 2 | Simular back-pressure en el pool |
| 🎯 Quiz | C10K, epoll, io_uring |

## 📚 Parte 1: El Protocolo HTTP/1.1

### Formato de la petición:
```
GET /index.html HTTP/1.1

Host: localhost:8080

User-Agent: curl/7.88.1

Accept: */*

Connection: keep-alive



```

### Formato de la respuesta:
```
HTTP/1.1 200 OK

Content-Type: text/html; charset=utf-8

Content-Length: 1234

Connection: close



<html>...</html>
```

### El ciclo de vida de una petición:

```
Cliente                          Servidor
  |                                  |
  |─── TCP SYN ──────────────────►  |
  |◄── TCP SYN-ACK ─────────────── |   (three-way handshake)
  |─── TCP ACK ──────────────────► |
  |                                  |
  |─── GET /path HTTP/1.1

 ►|  recv()
  |                                  |   → parsear método, path
  |                                  |   → leer archivo del disco
  |◄── HTTP/1.1 200 OK
... ───── |  send()
  |                                  |
  |─── FIN ──────────────────────► |  close()
```

In [ ]:
# 🔨 SIMULACIÓN: El Parser HTTP en Python
# Equivalente al http_parse_request() de http_server.c

import re
from dataclasses import dataclass, field
from typing import Dict, Optional

@dataclass
class HttpRequest:
    method: str = ""
    path: str = ""
    version: str = ""
    headers: Dict[str, str] = field(default_factory=dict)
    body: bytes = b""
    
    @property
    def keep_alive(self) -> bool:
        conn = self.headers.get("connection", "").lower()
        if conn == "keep-alive": return True
        if conn == "close": return False
        return self.version == "HTTP/1.1"

def parse_http_request(raw: bytes) -> Optional[HttpRequest]:
    """
    Parsea una petición HTTP raw (bytes).
    Equivalente a http_parse_request() en C.
    """
    # Separar cabeceras del cuerpo (separadas por \r\n\r\n)
    if b"\r\n\r\n" not in raw:
        return None
    
    header_part, _, body = raw.partition(b"\r\n\r\n")
    lines = header_part.decode("utf-8", errors="replace").split("\r\n")
    
    if not lines:
        return None
    
    # Línea de petición: "GET /path HTTP/1.1"
    partes = lines[0].split(" ", 2)
    if len(partes) != 3:
        return None
    
    req = HttpRequest(method=partes[0], path=partes[1], version=partes[2])
    
    # Parsear cabeceras
    for line in lines[1:]:
        if ":" in line:
            key, _, val = line.partition(":")
            req.headers[key.strip().lower()] = val.strip()
    
    # Cuerpo (si hay Content-Length)
    content_length = int(req.headers.get("content-length", 0))
    req.body = body[:content_length]
    
    return req

# Tests
peticiones = [
    b"GET /index.html HTTP/1.1\r\nHost: localhost\r\nConnection: keep-alive\r\n\r\n",
    b"POST /api/data HTTP/1.0\r\nContent-Length: 5\r\n\r\nhello",
    b"DELETE /resource HTTP/1.1\r\n\r\n",
]

for raw in peticiones:
    req = parse_http_request(raw)
    if req:
        print(f"  {req.method} {req.path} → keep_alive={req.keep_alive} | body={req.body}")
    else:
        print("  ERROR: petición inválida")

In [ ]:
# 🔨 IMPLEMENTACIÓN: Thread Pool en Python
# Modela la arquitectura de thread_pool_init/submit/destroy en C

import threading
import queue
import time
from typing import Callable

class ThreadPool:
    """
    Pool de hilos para procesar conexiones HTTP concurrentemente.
    Modela la arquitectura del thread pool de http_server.c.
    
    C:                              Python:
    pthread_mutex_t mutex           threading.Lock()
    pthread_cond_t cond_not_empty   threading.Condition()
    WorkItem *queue_head            queue.Queue(maxsize=capacity)
    pthread_create(worker_fn)       threading.Thread(target=worker)
    """
    def __init__(self, num_workers: int, capacity: int = 256):
        self.num_workers = num_workers
        self._queue = queue.Queue(maxsize=capacity)
        self._workers = []
        self._running = True
        
        for _ in range(num_workers):
            t = threading.Thread(target=self._worker_loop, daemon=True)
            t.start()
            self._workers.append(t)
    
    def _worker_loop(self):
        """Equivalente a worker_thread() en C."""
        while self._running:
            try:
                handler, client_fd = self._queue.get(timeout=0.1)
                try:
                    handler(client_fd)
                except Exception as e:
                    print(f"  [ERROR] Worker: {e}")
                finally:
                    self._queue.task_done()
            except queue.Empty:
                continue
    
    def submit(self, handler: Callable, client_fd: int) -> bool:
        """
        Equivalente a thread_pool_submit() en C.
        Retorna False si la cola está llena (back-pressure).
        """
        try:
            self._queue.put_nowait((handler, client_fd))
            return True
        except queue.Full:
            return False
    
    def shutdown(self, wait=True):
        """Equivalente a thread_pool_destroy() en C."""
        self._running = False
        if wait:
            for w in self._workers:
                w.join(timeout=5.0)

# Demo: simular 10 conexiones procesadas por 3 workers
def procesar_conexion(fd: int):
    """Simula procesar una petición HTTP (lectura + respuesta)."""
    time.sleep(0.01)  # 10ms simulados
    print(f"  [Worker {threading.current_thread().name}] fd={fd} procesado")

pool = ThreadPool(num_workers=3, capacity=20)
print("Enviando 10 'conexiones' al pool (3 workers):")
for i in range(10):
    ok = pool.submit(procesar_conexion, client_fd=i)
    if not ok:
        print(f"  fd={i}: RECHAZADO (cola llena)")

time.sleep(0.2)  # Dar tiempo a los workers
pool.shutdown(wait=False)
print("Pool apagado.")

## 🏗️ Parte 3: Arquitectura Real de http_server.c

```c
// Ciclo principal del servidor (simplificado)
int http_server_run(HttpServer *server) {
    while (server->running) {
        // 1. Esperar nueva conexión TCP
        int client_fd = accept(server->listen_fd, NULL, NULL);
        
        // 2. Encolar en el thread pool
        if (thread_pool_submit(&server->pool, client_fd) < 0) {
            // Cola llena → respuesta 503
            send(client_fd, "HTTP/1.1 503 ...", ..., MSG_NOSIGNAL);
            close(client_fd);
        }
    }
    return 0;
}

// Worker del thread pool (un hilo por worker)
static void *worker_thread(void *arg) {
    ThreadPool *pool = arg;
    while (1) {
        pthread_mutex_lock(&pool->mutex);
        
        // Esperar hasta que haya trabajo
        while (pool->queue_size == 0 && pool->running)
            pthread_cond_wait(&pool->cond_not_empty, &pool->mutex);
        
        if (!pool->running && pool->queue_size == 0) {
            pthread_mutex_unlock(&pool->mutex);
            return NULL;  // Salida limpia
        }
        
        // Desencolar y procesar fuera del mutex
        WorkItem *item = dequeue(pool);
        pthread_cond_signal(&pool->cond_not_full);
        pthread_mutex_unlock(&pool->mutex);
        
        handle_client(server, item->client_fd);
        free(item);
    }
}
```

**¿Por qué liberar el mutex ANTES de procesar la petición?**
> Si procesáramos con el mutex tomado, solo un worker podría trabajar a la vez.
> El mutex solo protege la cola, no el procesamiento de la petición.

## ✏️ Ejercicio 1: Parser de Cabeceras con Validación

**Problema:** Extiende `parse_http_request` para:
1. Rechazar métodos no soportados (solo GET, POST, PUT, DELETE)
2. Rechazar rutas con `..` (directory traversal)
3. Extraer el `Content-Type` de las cabeceras de petición

In [ ]:
# ✏️ EJERCICIO 1

METODOS_SOPORTADOS = {"GET", "POST", "PUT", "DELETE"}

def parse_http_request_seguro(raw: bytes) -> tuple:
    """
    Parsea y valida la petición HTTP.
    
    Returns:
        (HttpRequest, None) si es válida
        (None, str_error) si es inválida
    """
    # TODO: Implementar
    # 1. Parsear con parse_http_request()
    # 2. Validar método
    # 3. Validar que path no contiene ".."
    # 4. Retornar (req, None) o (None, "descripción del error")
    pass

# Tests
casos = [
    (b"GET / HTTP/1.1\r\n\r\n", True),
    (b"HACK / HTTP/1.1\r\n\r\n", False),
    (b"GET /../../etc/passwd HTTP/1.1\r\n\r\n", False),
    (b"POST /api HTTP/1.1\r\nContent-Type: application/json\r\n\r\n{}", True),
]

for raw, esperado_ok in casos:
    req, error = parse_http_request_seguro(raw) or (None, "not implemented")
    estado = "OK" if (req is not None) == esperado_ok else "FAIL"
    print(f"  [{estado}] {raw[:40]!r}... → {error or 'válida'}")

In [ ]:
# 💡 SOLUCIÓN — Ejercicio 1

def parse_http_request_seguro(raw: bytes) -> tuple:
    req = parse_http_request(raw)
    if req is None:
        return None, "Formato de petición inválido"
    
    if req.method not in METODOS_SOPORTADOS:
        return None, f"Método no soportado: {req.method}"
    
    if ".." in req.path:
        return None, f"Path peligroso (directory traversal): {req.path}"
    
    return req, None

print("Validación de peticiones HTTP:")
casos = [
    (b"GET / HTTP/1.1\r\n\r\n", True),
    (b"HACK / HTTP/1.1\r\n\r\n", False),
    (b"GET /../../etc/passwd HTTP/1.1\r\n\r\n", False),
    (b"POST /api HTTP/1.1\r\nContent-Type: application/json\r\n\r\n{}", True),
    (b"DELETE /resource HTTP/1.1\r\n\r\n", True),
]
for raw, esperado_ok in casos:
    req, error = parse_http_request_seguro(raw)
    estado = "✓" if (req is not None) == esperado_ok else "✗"
    print(f"  [{estado}] {raw[:45].decode()!r} → {error or '✅ válida'}")

## ✏️ Ejercicio 2: Back-Pressure y Graceful Degradation

**Problema:** Simula un escenario donde el servidor está bajo carga y la cola
se llena. El servidor debe responder con 503 en lugar de perder peticiones.

In [ ]:
# ✏️ EJERCICIO 2

def simular_servidor_bajo_carga(num_peticiones=50, workers=2, capacidad_cola=5):
    """
    Simula el comportamiento del servidor cuando la cola está casi llena.
    Peticiones que no caben → respuesta 503 (back-pressure).
    
    Returns:
        dict con 'aceptadas', 'rechazadas' y 'procesadas'
    """
    # TODO: Implementar usando ThreadPool
    # Crear pool con los parámetros dados
    # Enviar num_peticiones
    # Contar aceptadas vs rechazadas
    pass

resultado = simular_servidor_bajo_carga()
print("Resultado:", resultado)

In [ ]:
# 💡 SOLUCIÓN — Ejercicio 2

def simular_servidor_bajo_carga(num_peticiones=50, workers=2, capacidad_cola=5):
    procesadas = []
    lock = threading.Lock()
    
    def handler_lento(fd):
        time.sleep(0.02)   # 20ms de procesamiento
        with lock:
            procesadas.append(fd)
    
    pool = ThreadPool(num_workers=workers, capacity=capacidad_cola)
    
    aceptadas = rechazadas = 0
    for i in range(num_peticiones):
        if pool.submit(handler_lento, client_fd=i):
            aceptadas += 1
        else:
            rechazadas += 1
    
    time.sleep(0.5)   # Dar tiempo a que terminen
    pool.shutdown(wait=False)
    
    return {
        "enviadas": num_peticiones,
        "aceptadas": aceptadas,
        "rechazadas": rechazadas,
        "procesadas": len(procesadas),
    }

resultado = simular_servidor_bajo_carga(num_peticiones=50, workers=2, capacidad_cola=5)
print("Resultado del test de back-pressure:")
for k, v in resultado.items():
    print(f"  {k}: {v}")
print(f"  → {resultado['rechazadas']} peticiones respondidas con 503")

## 🎯 Quiz — Preguntas de Entrevista

**P1:** ¿Cuál es el problema C10K y cómo se resuelve con epoll?
> **R:** C10K = 10,000 conexiones concurrentes. El modelo thread-per-connection
> no escala (cada hilo necesita ~8MB de stack). epoll() monitorea miles de sockets
> en un solo hilo con I/O event-driven: solo "despierta" cuando hay datos listos.

**P2:** ¿Por qué usar `MSG_NOSIGNAL` en `send()`?
> **R:** Sin `MSG_NOSIGNAL`, enviar a un socket cerrado genera SIGPIPE que
> mata el proceso. Con `MSG_NOSIGNAL`, `send()` retorna -1 con errno=EPIPE,
> que podemos manejar limpiamente.

**P3:** ¿Qué es io_uring y cómo mejora sobre epoll?
> **R:** io_uring (Linux 5.1+) usa un ring buffer en shared memory entre
> kernel y userspace para enviar múltiples operaciones I/O en una sola syscall.
> Elimina el overhead de syscalls por operación y permite I/O completamente async.

**P4:** ¿Qué es el problema de la "thundering herd" con accept()?
> **R:** Con SO_REUSEPORT o múltiples hilos bloqueados en accept(), una nueva
> conexión puede despertar a TODOS los hilos aunque solo uno puede aceptarla.
> Linux moderno mitiga esto con EPOLLEXCLUSIVE en epoll_ctl.